# Step 0 : About Project

In [90]:
"""
=====================================================================
ENTERPRISE DATA QUALITY & BUSINESS UNDERSTANDING FRAMEWORK
=====================================================================

Project Overview
----------------
This Jupyter Notebook is designed to generate enterprise-style Excel
reports for data analysis, data quality validation, and business
understanding activities.

The framework automates multiple reporting and validation processes
commonly used in real-world enterprise data engineering, analytics,
and governance projects.

Main Modules
------------
1. Data Quality Checks
   - Completeness checks
   - Uniqueness checks
   - Validity checks
   - Consistency checks
   - KPI summaries
   - Executive dashboard generation

2. Business Understanding Framework
   - Customer distribution analysis
   - Country and industry analysis
   - SAP vs LUCI comparison
   - Duplicate and primary key analysis
   - Hierarchy and segmentation reporting
   - Business KPI generation

3. Project Overview Reporting
   - Project metadata
   - Dataset statistics
   - Summary insights
   - Structured reporting sheets

Key Features
-------------
- Enterprise-style Excel formatting
- Automated workbook and worksheet generation
- Styled KPI cards and dashboard sections
- SQL + Pandas based analytical processing
- OpenPyXL-based advanced Excel formatting
- Reusable and modular framework structure
- Consistent workbook behavior across modules

Technologies Used
-----------------
- Python
- Pandas
- SQLite
- OpenPyXL
- Jupyter Notebook

Design Goal
------------
Maintain a consistent, scalable, and reusable reporting framework
that produces standardized enterprise-quality Excel outputs across
all project modules.

Output
------
The notebook generates formatted Excel workbooks containing:
- Data Quality Dashboard
- Business Understanding Reports
- Project Overview Sheets
- KPI Summaries
- Analytical Tables and Insights

=====================================================================
"""

'\n=====================================================================\nENTERPRISE DATA QUALITY & BUSINESS UNDERSTANDING FRAMEWORK\n=====================================================================\n\nProject Overview\n----------------\nThis Jupyter Notebook is designed to generate enterprise-style Excel\nreports for data analysis, data quality validation, and business\nunderstanding activities.\n\nThe framework automates multiple reporting and validation processes\ncommonly used in real-world enterprise data engineering, analytics,\nand governance projects.\n\nMain Modules\n------------\n1. Data Quality Checks\n   - Completeness checks\n   - Uniqueness checks\n   - Validity checks\n   - Consistency checks\n   - KPI summaries\n   - Executive dashboard generation\n\n2. Business Understanding Framework\n   - Customer distribution analysis\n   - Country and industry analysis\n   - SAP vs LUCI comparison\n   - Duplicate and primary key analysis\n   - Hierarchy and segmentation report

# Step - 1 : Install Libraries

## 1.1 Install Commands

In [91]:
#!pip install pandas
#!pip install openpyxl

# Step - 2 : Import Libraries

## 2.1 Core Imports

In [92]:
import os
import re
import uuid
import sqlite3
import pandas as pd
import time

from openpyxl import (
    Workbook,
    load_workbook
)

from openpyxl.styles import (
    Font,
    PatternFill,
    Border,
    Side,
    Alignment
)

from openpyxl.utils import (
    get_column_letter
)

from openpyxl.worksheet.table import (
    Table,
    TableStyleInfo
)

# Step - 3 : Load Input / Output / Runtime Objects

## 3.1 Load Dataset and Prepare SQLite

In [93]:
start_time = time.perf_counter()
# =========================================================
# STEP 1 : FILE PATHS
# =========================================================

input_file_path = "input_files/customer_sample_500.csv.xlsx"

output_file_path = "output_files/QualityCheck_results.xlsx"

# =========================================================
# STEP 2 : CREATE OUTPUT DIRECTORY
# =========================================================

os.makedirs(
    "output_files",
    exist_ok=True
)

# =========================================================
# STEP 3 : LOAD INPUT DATA
# =========================================================

df = pd.read_excel(
    input_file_path,
    dtype=str
)

# =========================================================
# STEP 4 : HANDLE NULL VALUES
# =========================================================

df = df.fillna("")

# =========================================================
# STEP 5 : SQLITE DATABASE
# =========================================================

conn = sqlite3.connect(":memory:")

df.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

cursor = conn.cursor()

# =========================================================
# STEP 6 : CREATE / LOAD WORKBOOK
# =========================================================

if os.path.exists(output_file_path):

    wb = load_workbook(output_file_path)

else:

    wb = Workbook()

# =========================================================
# STEP 7 : REMOVE DEFAULT / OLD SHEETS
# =========================================================

remove_sheets = [

    "Sheet",

    "Data Quality Checks",
    "Data_Quality_Checks",

    "Business Framework",
    "Business_Framework",

    "Project Overview",
    "Project_Overview"

]

for sheet in remove_sheets:

    if sheet in wb.sheetnames:

        std = wb[sheet]

        wb.remove(std)

# =========================================================
# STEP 8 : CREATE REQUIRED SHEETS
# =========================================================

required_sheets = [

    "Project_Overview",
    "Data_Quality_Checks",
    "Business_Framework"

]

for sheet in required_sheets:

    if sheet not in wb.sheetnames:

        wb.create_sheet(title=sheet)

# =========================================================
# STEP 9 : SHEET REFERENCES
# =========================================================

overview_ws = wb["Project_Overview"]

dq_ws = wb["Data_Quality_Checks"]

business_ws = wb["Business_Framework"]

# =========================================================
# STEP 10 : INITIAL TITLES
# =========================================================

title_fill = PatternFill(
    "solid",
    fgColor="1F3A5F"
)

title_font = Font(
    bold=True,
    size=16,
    color="FFFFFF"
)

center = Alignment(
    horizontal="center",
    vertical="center"
)

sheet_titles = {

    "Project_Overview":
        "PROJECT OVERVIEW",

    "Data_Quality_Checks":
        "DATA QUALITY CHECKS",

    "Business_Framework":
        "BUSINESS UNDERSTANDING FRAMEWORK"
}

for sheet_name, title in sheet_titles.items():

    ws = wb[sheet_name]

    ws.merge_cells("A1:Y2")

    cell = ws["A1"]

    cell.value = title

    cell.font = title_font

    cell.fill = title_fill

    cell.alignment = center

    ws.sheet_view.showGridLines = False

# =========================================================
# STEP 11 : SAVE INITIAL WORKBOOK
# =========================================================

wb.save(output_file_path)

# =========================================================
# STEP 12 : VALIDATION
# =========================================================

print("=" * 60)

print("WORKBOOK CREATED SUCCESSFULLY")

print("=" * 60)

print(
    "Input File :",
    os.path.abspath(input_file_path)
)

print(
    "Output File :",
    os.path.abspath(output_file_path)
)

print(
    "Rows :",
    len(df)
)

print(
    "Columns :",
    len(df.columns)
)

print("=" * 60)
print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

WORKBOOK CREATED SUCCESSFULLY
Input File : c:\Users\U1109200\Downloads\INTERN_TASK_REPO\FrameWork\input_files\customer_sample_500.csv.xlsx
Output File : c:\Users\U1109200\Downloads\INTERN_TASK_REPO\FrameWork\output_files\QualityCheck_results.xlsx
Rows : 500
Columns : 55
Execution Time: 0.21 seconds


# Step - 4 : DQ Checks (Each Check in Separate Cell)

## 4.1 Prepare DQ Base Data

In [94]:
start_time = time.perf_counter()
# =========================================================
# STEP 4.1 : SHARED DQ HELPERS
# =========================================================

# WORKING COPY

data = df.copy()

# =========================================================
# STANDARDIZE ALL COLUMNS
# =========================================================

for col in data.columns:

    data[col] = (
        data[col]
        .astype(str)
        .fillna("")
        .str.strip()
    )

# =========================================================
# TOTAL RECORD COUNT
# =========================================================

total_records = len(data)

# =========================================================
# BLANK VALUE CHECKER
# =========================================================

def is_blank(series: pd.Series) -> pd.Series:

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

# =========================================================
# SAFE PERCENTAGE FUNCTION
# =========================================================

def pct(
    numerator: int,
    denominator: int
) -> str:

    if denominator == 0:

        return "0.0%"

    return (
        f"{round((numerator / denominator) * 100, 2)}%"
    )

# =========================================================
# DQ SHEET REFERENCE
# =========================================================

ws = dq_ws

print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

Execution Time: 0.02 seconds


## 4.2 Completeness Check

In [95]:
start_time = time.perf_counter()
# DQ Check 1: Completeness
completeness_rows = []
total_missing = 0
for col in data.columns:
    s = data[col].fillna("").astype(str)
    missing_count = int(s.str.strip().eq("").sum())
    total_missing += missing_count
    completeness_rows.append(
        {
            "Column": col,
            "Missing_Values": missing_count,
            "Completeness_%": pct(total_records - missing_count, total_records),
        }
    )
completeness_df = pd.DataFrame(completeness_rows)
total_cells = total_records * len(data.columns)
completeness_pivot = pd.DataFrame(
    {"Status": ["Missing", "Non-Missing"], "Count": [total_missing, total_cells - total_missing]}
)

print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

Execution Time: 0.02 seconds


## 4.3 Uniqueness Check

In [96]:
start_time = time.perf_counter()
# DQ Check 2: Uniqueness
uniqueness_rows = []
total_duplicates = 0
for col in data.columns:
    s = data[col].fillna("").astype(str).str.strip()
    non_blank = s[s.ne("")]
    distinct_count = int(non_blank.nunique(dropna=True))
    duplicate_count = int(max(len(non_blank) - distinct_count, 0))
    total_duplicates += duplicate_count
    uniqueness_rows.append(
        {
            "Column": col,
            "Distinct_Values": distinct_count,
            "Duplicate_Values": duplicate_count,
            "Uniqueness_%": pct(distinct_count, len(non_blank)),
        }
    )
uniqueness_df = pd.DataFrame(uniqueness_rows)
print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

Execution Time: 0.02 seconds


## 4.4 Validity Check

In [97]:
start_time = time.perf_counter()
# DQ Check 3: Validity
siret_blank = is_blank(data["SIRET"]) if "SIRET" in data.columns else pd.Series([True] * total_records)
siren_blank = is_blank(data["SIREN"]) if "SIREN" in data.columns else pd.Series([True] * total_records)
country_blank = is_blank(data["COUNTRY_CD"]) if "COUNTRY_CD" in data.columns else pd.Series([True] * total_records)

invalid_siret_mask = (~siret_blank) & (~data["SIRET"].fillna("").astype(str).str.strip().str.fullmatch(r"\d{14}")) if "SIRET" in data.columns else pd.Series([False] * total_records)
invalid_siren_mask = (~siren_blank) & (~data["SIREN"].fillna("").astype(str).str.strip().str.fullmatch(r"\d{9}")) if "SIREN" in data.columns else pd.Series([False] * total_records)
invalid_country_mask = (~country_blank) & (~data["COUNTRY_CD"].fillna("").astype(str).str.strip().str.fullmatch(r"[A-Z]{2}")) if "COUNTRY_CD" in data.columns else pd.Series([False] * total_records)

invalid_siret = int(invalid_siret_mask.sum())
invalid_siren = int(invalid_siren_mask.sum())
invalid_country = int(invalid_country_mask.sum())

validity_df = pd.DataFrame([
    {"Validity_Check": "Invalid_SIRET", "Invalid_Count": invalid_siret, "Invalid_%": pct(invalid_siret, total_records)},
    {"Validity_Check": "Invalid_SIREN", "Invalid_Count": invalid_siren, "Invalid_%": pct(invalid_siren, total_records)},
    {"Validity_Check": "Invalid_Country_Code", "Invalid_Count": invalid_country, "Invalid_%": pct(invalid_country, total_records)},
])

any_invalid_mask = invalid_siret_mask | invalid_siren_mask | invalid_country_mask
total_invalid_rows = int(any_invalid_mask.sum())
validity_pivot = pd.DataFrame({"Status": ["Invalid", "Valid"], "Count": [total_invalid_rows, total_records - total_invalid_rows]})
print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

Execution Time: 0.01 seconds


## 4.5 Consistency Check

In [98]:
start_time = time.perf_counter()
# DQ Check 4: Consistency
has_country_cols = "COUNTRY_CD" in data.columns and "LUCI_COUNTRY_CD" in data.columns
has_city_cols = "SAP_CITY" in data.columns and "LUCI_CITY" in data.columns

if has_country_cols:
    left_country = data["COUNTRY_CD"].fillna("").astype(str).str.strip()
    right_country = data["LUCI_COUNTRY_CD"].fillna("").astype(str).str.strip()
    valid_country_pair = left_country.ne("") & right_country.ne("")
    country_mismatch_mask = valid_country_pair & left_country.ne(right_country)
else:
    country_mismatch_mask = pd.Series([False] * total_records)

if has_city_cols:
    left_city = data["SAP_CITY"].fillna("").astype(str).str.strip().str.upper()
    right_city = data["LUCI_CITY"].fillna("").astype(str).str.strip().str.upper()
    valid_city_pair = left_city.ne("") & right_city.ne("")
    city_mismatch_mask = valid_city_pair & left_city.ne(right_city)
else:
    city_mismatch_mask = pd.Series([False] * total_records)

country_mismatch = int(country_mismatch_mask.sum())
city_mismatch = int(city_mismatch_mask.sum())

consistency_df = pd.DataFrame([
    {"Consistency_Check": "Country_Mismatch", "Mismatch_Count": country_mismatch, "Mismatch_%": pct(country_mismatch, total_records)},
    {"Consistency_Check": "City_Mismatch", "Mismatch_Count": city_mismatch, "Mismatch_%": pct(city_mismatch, total_records)},
])

any_mismatch_mask = country_mismatch_mask | city_mismatch_mask
total_mismatch_rows = int(any_mismatch_mask.sum())
consistency_pivot = pd.DataFrame({"Status": ["Mismatch", "Matched"], "Count": [total_mismatch_rows, total_records - total_mismatch_rows]})

print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

Execution Time: 0.00 seconds


## 4.6 Write DQ Output

### 4.6.1 Build Data Quality Checks Sheet

In [99]:
start_time = time.perf_counter()
# =========================================================
# DQ OUTPUT WRITER - PART 1
# =========================================================

error_cells = total_missing + total_duplicates

quality_score = round(
    100 - (
        (error_cells / max(total_cells, 1)) * 100
    ),
    2
)

# =========================================================
# LOAD EXISTING WORKBOOK
# =========================================================

file_path = output_file_path

wb = load_workbook(file_path)

# =========================================================
# USE EXISTING DQ SHEET
# =========================================================

ws = wb["Data_Quality_Checks"]

# =========================================================
# REMOVE OLD MERGED CELLS FIRST
# =========================================================

merged_ranges = list(ws.merged_cells.ranges)

for merged in merged_ranges:

    ws.unmerge_cells(str(merged))

# =========================================================
# CLEAR OLD CONTENT
# =========================================================

for row in ws.iter_rows():

    for cell in row:

        cell.value = None

# =========================================================
# SHEET SETTINGS
# =========================================================

ws.sheet_view.showGridLines = False

ws.freeze_panes = None

ws.sheet_view.zoomScale = 90

# =========================================================
# COLORS
# =========================================================

dark = "1F3A5F"

mid = "2F75B5"

light = "EAF2FB"

light_alt = "F8FBFF"

white = "FFFFFF"

text_dark = "1A1A1A"

green = "2E7D32"

orange = "E67E22"

red = "C0392B"

# =========================================================
# FONTS
# =========================================================

title_font = Font(
    name="Calibri",
    size=18,
    bold=True,
    color=white
)

kpi_font = Font(
    name="Calibri",
    size=13,
    bold=True,
    color=white
)

section_font = Font(
    name="Calibri",
    size=12,
    bold=True,
    color=white
)

header_font = Font(
    name="Calibri",
    size=10,
    bold=True,
    color=text_dark
)

body_font = Font(
    name="Calibri",
    size=10,
    bold=False,
    color=text_dark
)

# =========================================================
# ALIGNMENTS
# =========================================================

center = Alignment(
    horizontal="center",
    vertical="center",
    wrap_text=True
)

left = Alignment(
    horizontal="left",
    vertical="center",
    wrap_text=True
)

# =========================================================
# BORDER
# =========================================================

thin = Side(
    style="thin",
    color="D0D7DE"
)

border = Border(
    left=thin,
    right=thin,
    top=thin,
    bottom=thin
)

# =========================================================
# MAIN TITLE
# =========================================================

ws.merge_cells("A1:AH2")

c = ws["A1"]

c.value = "ENTERPRISE DATA QUALITY DASHBOARD"

c.fill = PatternFill(
    fill_type="solid",
    start_color=dark,
    end_color=dark
)

c.font = title_font

c.alignment = center

# =========================================================
# KPI CARDS
# =========================================================

kpis = [

    (
        "TOTAL RECORDS",
        f"{total_records:,}",
        "A4:G6",
        green
    ),

    (
        "MISSING VALUES",
        f"{total_missing:,}",
        "J4:P6",
        orange
    ),

    (
        "INVALID ROWS",
        f"{total_invalid_rows:,}",
        "S4:Y6",
        red
    ),

    (
        "QUALITY SCORE",
        f"{quality_score}%",
        "AB4:AH6",
        mid
    )

]

for label, value, cell_range, color in kpis:

    ws.merge_cells(cell_range)

    start_cell = ws[
        cell_range.split(":")[0]
    ]

    start_cell.value = (
        f"{label}\n{value}"
    )

    start_cell.fill = PatternFill(
        fill_type="solid",
        start_color=color,
        end_color=color
    )

    start_cell.font = kpi_font

    start_cell.alignment = center

    start_row = int(
        re.sub(
            r"[A-Z]",
            "",
            cell_range.split(":")[0]
        )
    )

    end_row = int(
        re.sub(
            r"[A-Z]",
            "",
            cell_range.split(":")[1]
        )
    )

    start_col = ws[
        cell_range.split(":")[0]
    ].column

    end_col = ws[
        cell_range.split(":")[1]
    ].column

    for r in range(
        start_row,
        end_row + 1
    ):

        for col_idx in range(
            start_col,
            end_col + 1
        ):

            ws.cell(
                r,
                col_idx
            ).border = border

# =========================================================
# ROW HEIGHTS
# =========================================================

ws.row_dimensions[1].height = 34

ws.row_dimensions[2].height = 30

ws.row_dimensions[4].height = 26

ws.row_dimensions[5].height = 26

ws.row_dimensions[6].height = 26

print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

Execution Time: 0.02 seconds


### 4.6.2 DQ Section Writers

In [100]:
start_time = time.perf_counter()
def write_table(start_col, header_row, data_row, table_df, first_col_left=True):
    if table_df is None or table_df.empty:
        return 0

    for idx, col_name in enumerate(table_df.columns):
        cell = ws.cell(header_row, start_col + idx)
        cell.value = col_name
        cell.fill = PatternFill(fill_type="solid", start_color=light, end_color=light)
        cell.font = header_font
        cell.alignment = center
        cell.border = border

    for i in range(len(table_df)):
        row_num = data_row + i
        alt_fill = PatternFill(
            fill_type="solid",
            start_color=light_alt if i % 2 else white,
            end_color=light_alt if i % 2 else white,
        )
        for j, col_name in enumerate(table_df.columns):
            value = table_df.iloc[i][col_name]
            cell = ws.cell(row_num, start_col + j)
            cell.value = value
            cell.font = body_font
            cell.alignment = left if (first_col_left and j == 0) else center
            cell.fill = alt_fill
            cell.border = border
        ws.row_dimensions[row_num].height = 22

    return len(table_df)

def write_section(start_col, title, main_df, status_df):
    title_row = 9
    header_row = 11
    data_row = 12

    title_start = ws.cell(title_row, start_col)
    title_end_col = start_col + 6
    ws.merge_cells(start_row=title_row, start_column=start_col, end_row=title_row, end_column=title_end_col)
    title_start.value = title
    title_start.fill = PatternFill(fill_type="solid", start_color=dark, end_color=dark)
    title_start.font = section_font
    title_start.alignment = center
    title_start.border = border

    main_rows = write_table(start_col, header_row, data_row, main_df, first_col_left=True)
    status_rows = write_table(start_col + 5, header_row, data_row, status_df, first_col_left=True)

    if main_rows == 0 and status_rows == 0:
        note = ws.cell(data_row, start_col)
        note.value = "No records to display"
        note.font = body_font
        note.alignment = center
        note.fill = PatternFill(fill_type="solid", start_color=white, end_color=white)
        note.border = border

write_section(1, "COMPLETENESS CHECK", completeness_df, completeness_pivot)
write_section(10, "UNIQUENESS CHECK", uniqueness_df, pd.DataFrame({"Status": ["Duplicate Cells", "Unique Cells"], "Count": [total_duplicates, total_cells - total_duplicates]}))
write_section(19, "VALIDITY CHECK", validity_df, validity_pivot)
write_section(28, "CONSISTENCY CHECK", consistency_df, consistency_pivot)
print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

Execution Time: 0.05 seconds


### 4.6.3 DQ Layout and Save

In [101]:
start_time = time.perf_counter()
for start_col in [1, 10, 19, 28]:
    for offset in range(4):
        col_idx = start_col + offset
        col_letter = get_column_letter(col_idx)
        max_len = len(str(ws.cell(11, col_idx).value or ""))
        for r in range(12, ws.max_row + 1):
            value = ws.cell(r, col_idx).value
            if value not in (None, ""):
                max_len = max(max_len, len(str(value)))
        if offset == 0:
            ws.column_dimensions[col_letter].width = min(max(max_len + 3, 22), 36)
        else:
            ws.column_dimensions[col_letter].width = min(max(max_len + 3, 12), 22)

    ws.column_dimensions[get_column_letter(start_col + 4)].width = 3

    for offset in [5, 6]:
        col_idx = start_col + offset
        col_letter = get_column_letter(col_idx)
        max_len = len(str(ws.cell(11, col_idx).value or ""))
        for r in range(12, ws.max_row + 1):
            value = ws.cell(r, col_idx).value
            if value not in (None, ""):
                max_len = max(max_len, len(str(value)))
        ws.column_dimensions[col_letter].width = min(max(max_len + 3, 12), 20)

for col_letter in ["H", "I", "Q", "R", "Z", "AA"]:
    ws.column_dimensions[col_letter].width = 3

wb.save(file_path)
print("DQ output file:", os.path.abspath(file_path))
print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

DQ output file: c:\Users\U1109200\Downloads\INTERN_TASK_REPO\FrameWork\output_files\QualityCheck_results.xlsx
Execution Time: 0.02 seconds


# Step - 5 : Code for Business Framework Sheet

## 5.1 Create Business Framework Function

In [102]:
start_time = time.perf_counter()
# =========================================================
# FOUNDATION TABLES
# =========================================================

def build_foundation_tables(cursor):

    total_records = cursor.execute(
        "SELECT COUNT(*) FROM customers"
    ).fetchone()[0]

    pk_null_count = cursor.execute(
        """
        SELECT COUNT(*)
        FROM customers
        WHERE PK_CUSTOMER_ID IS NULL
        OR TRIM(PK_CUSTOMER_ID) = ''
        """
    ).fetchone()[0]

    pk_non_null_count = total_records - pk_null_count

    distinct_values = cursor.execute(
        """
        SELECT COUNT(DISTINCT PK_CUSTOMER_ID)
        FROM customers
        """
    ).fetchone()[0]

    duplicate_values = total_records - distinct_values

    pk_table = pd.DataFrame({

        "PK_Status": [
            "Null",
            "Non-Null"
        ],

        "Count": [
            pk_null_count,
            pk_non_null_count
        ],

        "Percentage": [
            f"{round((pk_null_count / total_records) * 100, 1)}%",
            f"{round((pk_non_null_count / total_records) * 100, 1)}%"
        ]
    })

    duplicate_table = pd.DataFrame({

        "Metric": [
            "Total Records",
            "Distinct Values",
            "Duplicate Values"
        ],

        "Count": [
            total_records,
            distinct_values,
            duplicate_values
        ],

        "Percentage": [
            "100.0%",
            f"{round((distinct_values / total_records) * 100, 1)}%",
            f"{round((duplicate_values / total_records) * 100, 1)}%"
        ]
    })

    return pk_table, duplicate_table


# =========================================================
# DISTRIBUTION TABLES
# =========================================================

def build_distribution_tables(conn):

    country_table = pd.read_sql_query(
        """
        SELECT
            COUNTRY_CD AS Country,
            COUNT(*) AS Record_Count,
            ROUND(
                COUNT(*) * 100.0 /
                (SELECT COUNT(*) FROM customers),
                1
            ) || '%' AS Percentage
        FROM customers
        GROUP BY COUNTRY_CD
        ORDER BY Record_Count DESC
        """,
        conn
    )

    industry_table = pd.read_sql_query(
        """
        SELECT
            INDUSTRY AS Industry,
            COUNT(*) AS Record_Count,
            ROUND(
                COUNT(*) * 100.0 /
                (SELECT COUNT(*) FROM customers),
                1
            ) || '%' AS Percentage
        FROM customers
        GROUP BY INDUSTRY
        ORDER BY Record_Count DESC
        """,
        conn
    )

    country_industry_table = pd.read_sql_query(
        """
        SELECT
            COUNTRY_CD AS Country,
            INDUSTRY AS Industry,
            COUNT(*) AS Record_Count,
            ROUND(
                COUNT(*) * 100.0 /
                (SELECT COUNT(*) FROM customers),
                1
            ) || '%' AS Percentage
        FROM customers
        GROUP BY COUNTRY_CD, INDUSTRY
        ORDER BY Country, Record_Count DESC
        """,
        conn
    )

    return (
        country_table,
        industry_table,
        country_industry_table
    )


# =========================================================
# LEGAL & SALES TABLES
# =========================================================

def build_legal_sales_tables(conn):

    legal_table = pd.read_sql_query(
        """
        SELECT
            LEGAL_STATUS_DESC AS Legal_Status,
            COUNT(*) AS Record_Count,
            ROUND(
                COUNT(*) * 100.0 /
                (SELECT COUNT(*) FROM customers),
                1
            ) || '%' AS Percentage
        FROM customers
        GROUP BY LEGAL_STATUS_DESC
        ORDER BY Record_Count DESC
        """,
        conn
    )

    sales_org_table = pd.read_sql_query(
        """
        SELECT
            SALES_ORG AS Sales_Org,
            COUNT(*) AS Record_Count,
            ROUND(
                COUNT(*) * 100.0 /
                (SELECT COUNT(*) FROM customers),
                1
            ) || '%' AS Percentage
        FROM customers
        GROUP BY SALES_ORG
        ORDER BY Record_Count DESC
        """,
        conn
    )

    return legal_table, sales_org_table
print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

Execution Time: 0.00 seconds


## 5.2 Create Distribution and Business Tables

In [103]:
start_time = time.perf_counter()
# =========================================================
# CUSTOMER TABLES
# =========================================================

def build_customer_tables(conn):

    customer_group_table = pd.read_sql_query(
        """
        SELECT
            CUSTOMER_GROUP_DESC AS Customer_Group,
            COUNT(*) AS Record_Count
        FROM customers
        GROUP BY CUSTOMER_GROUP_DESC
        ORDER BY Record_Count DESC
        """,
        conn
    )

    customer_nature_table = pd.read_sql_query(
        """
        SELECT
            CUSTOMER_NATURE_2_DESC AS Customer_Nature,
            COUNT(*) AS Record_Count
        FROM customers
        GROUP BY CUSTOMER_NATURE_2_DESC
        ORDER BY Record_Count DESC
        """,
        conn
    )

    customer_hierarchy_table = pd.read_sql_query(
        """
        SELECT
            CUSTOMER_GROUP_DESC AS Customer_Group,
            CUSTOMER_GRP_2_DESC AS Group_2,
            CUSTOMER_GRP_3_DESC AS Group_3,
            CUSTOMER_GRP_4_DESC AS Group_4,
            CUSTOMER_GRP_5_DESC AS Group_5,
            COUNT(*) AS Record_Count
        FROM customers
        GROUP BY
            CUSTOMER_GROUP_DESC,
            CUSTOMER_GRP_2_DESC,
            CUSTOMER_GRP_3_DESC,
            CUSTOMER_GRP_4_DESC,
            CUSTOMER_GRP_5_DESC
        """,
        conn
    )

    return (
        customer_group_table,
        customer_nature_table,
        customer_hierarchy_table
    )


# =========================================================
# SAP TABLES
# =========================================================

def build_sap_tables(conn):

    sap_city_legal_table = pd.read_sql_query(
        """
        SELECT
            SAP_CITY,
            SAP_LEGAL_NM,
            COUNT(*) AS Record_Count
        FROM customers
        GROUP BY SAP_CITY, SAP_LEGAL_NM
        """,
        conn
    )

    sap_country_city_table = pd.read_sql_query(
        """
        SELECT
            COUNTRY_CD,
            SAP_CITY,
            COUNT(*) AS Record_Count
        FROM customers
        GROUP BY COUNTRY_CD, SAP_CITY
        """,
        conn
    )

    sap_sales_org_table = pd.read_sql_query(
        """
        SELECT
            SALES_ORG,
            COUNT(*) AS Record_Count
        FROM customers
        GROUP BY SALES_ORG
        """,
        conn
    )

    return (
        sap_city_legal_table,
        sap_country_city_table,
        sap_sales_org_table
    )


# =========================================================
# LUCI TABLES
# =========================================================

def build_luci_tables(conn):

    luci_country_gers_table = pd.read_sql_query(
        """
        SELECT
            LUCI_COUNTRY_CD,
            LUCI_PDV_GERS,
            COUNT(*) AS Record_Count
        FROM customers
        GROUP BY LUCI_COUNTRY_CD, LUCI_PDV_GERS
        """,
        conn
    )

    luci_city_legal_table = pd.read_sql_query(
        """
        SELECT
            LUCI_CITY,
            LUCI_LEGAL_NM,
            COUNT(*) AS Record_Count
        FROM customers
        GROUP BY LUCI_CITY, LUCI_LEGAL_NM
        """,
        conn
    )

    luci_country_city_table = pd.read_sql_query(
        """
        SELECT
            LUCI_COUNTRY_CD,
            LUCI_CITY,
            COUNT(*) AS Record_Count
        FROM customers
        GROUP BY LUCI_COUNTRY_CD, LUCI_CITY
        """,
        conn
    )

    return (
        luci_country_gers_table,
        luci_city_legal_table,
        luci_country_city_table
    )
print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

Execution Time: 0.00 seconds


## 5.3 Create Rendering Functions

In [104]:
start_time = time.perf_counter()
# =========================================================
# BUSINESS FRAMEWORK SHEET SETUP + STYLING
# =========================================================

file_path = "output_files/QualityCheck_results.xlsx"

sheet_name = "Business_Framework"

# =========================================================
# LOAD WORKBOOK
# =========================================================

if os.path.exists(file_path):

    wb = load_workbook(file_path)

else:

    wb = Workbook()

# =========================================================
# REMOVE OLD BUSINESS SHEET
# =========================================================

if sheet_name in wb.sheetnames:

    del wb[sheet_name]

# =========================================================
# REMOVE DEFAULT SHEET
# =========================================================

if "Sheet" in wb.sheetnames:

    del wb["Sheet"]

# =========================================================
# CREATE FRESH BUSINESS SHEET
# =========================================================

ws = wb.create_sheet(title=sheet_name)

# =========================================================
# SHEET SETTINGS
# =========================================================

ws.sheet_view.showGridLines = False


# =========================================================
# COLORS
# =========================================================

dark = "1F3A5F"

light = "DCE6F1"

white = "FFFFFF"

# =========================================================
# BORDER
# =========================================================

thin = Side(
    style="thin",
    color="D0D7DE"
)

border = Border(
    left=thin,
    right=thin,
    top=thin,
    bottom=thin
)

# =========================================================
# ALIGNMENTS
# =========================================================

center = Alignment(
    horizontal="center",
    vertical="center"
)

left = Alignment(
    horizontal="left",
    vertical="center"
)

# =========================================================
# MAIN TITLE
# =========================================================

ws.merge_cells("A1:Y2")

title_cell = ws["A1"]

title_cell.value = "BUSINESS UNDERSTANDING FRAMEWORK"

title_cell.font = Font(
    bold=True,
    size=16,
    color=white
)

title_cell.fill = PatternFill(
    "solid",
    fgColor=dark
)

title_cell.alignment = center

# =========================================================
# WRITE TABLE FUNCTION
# =========================================================

def write_table(start_row, start_col, title, df):

    total_cols = len(df.columns)

    end_col = start_col + total_cols - 1

    # =====================================================
    # TABLE TITLE
    # =====================================================

    ws.merge_cells(
        start_row=start_row,
        start_column=start_col,
        end_row=start_row,
        end_column=end_col
    )

    title_cell = ws.cell(
        row=start_row,
        column=start_col
    )

    title_cell.value = title

    title_cell.font = Font(
        bold=True,
        color=white
    )

    title_cell.fill = PatternFill(
        "solid",
        fgColor=dark
    )

    title_cell.alignment = center

    # =====================================================
    # HEADERS
    # =====================================================

    header_row = start_row + 1

    for c, col_name in enumerate(df.columns):

        cell = ws.cell(
            row=header_row,
            column=start_col + c
        )

        cell.value = col_name

        cell.font = Font(
            bold=True
        )

        cell.fill = PatternFill(
            "solid",
            fgColor=light
        )

        cell.border = border

        cell.alignment = center

    # =====================================================
    # DATA
    # =====================================================

    data_start = start_row + 2

    for r in range(len(df)):

        for c, col_name in enumerate(df.columns):

            cell = ws.cell(
                row=data_start + r,
                column=start_col + c
            )

            value = df.iloc[r][col_name]

            cell.value = value

            cell.border = border

            if c == 0:

                cell.alignment = left

            else:

                cell.alignment = center

    # =====================================================
    # EXCEL FILTER TABLE
    # =====================================================

    table_ref = (
        f"{get_column_letter(start_col)}{header_row}:"
        f"{get_column_letter(end_col)}{data_start + len(df) - 1}"
    )

    table_name = (
        f"Table_{start_row}_{start_col}"
    )

    excel_table = Table(
        displayName=table_name,
        ref=table_ref
    )

    style = TableStyleInfo(
        name="TableStyleMedium2",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=False,
        showColumnStripes=False
    )

    excel_table.tableStyleInfo = style

    ws.add_table(excel_table)

    return data_start + len(df)

# =========================================================
# WRITE GROUP FUNCTION
# =========================================================

def write_group(row, heading, tables):

    # =====================================================
    # GROUP TITLE
    # =====================================================

    ws.merge_cells(
        start_row=row,
        start_column=1,
        end_row=row,
        end_column=18
    )

    group_cell = ws.cell(
        row=row,
        column=1
    )

    group_cell.value = heading

    group_cell.font = Font(
        bold=True,
        color=white
    )

    group_cell.fill = PatternFill(
        "solid",
        fgColor=dark
    )

    group_cell.alignment = center

    # =====================================================
    # HORIZONTAL TABLE LAYOUT
    # =====================================================

    current_col = 1

    table_row = row + 2

    max_end_row = table_row

    for title, df in tables:

        next_row = write_table(
            table_row,
            current_col,
            title,
            df
        )

        max_end_row = max(
            max_end_row,
            next_row
        )

        table_width = max(
            len(df.columns),
            3
        )

        # spacing between tables

        current_col += table_width + 1

    return max_end_row + 4
print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

Execution Time: 0.15 seconds


## 5.4 Create Business Framework Dashboard

In [105]:
start_time = time.perf_counter()
def business_understanding_framework(cursor, conn):

    pk_table, duplicate_table = build_foundation_tables(cursor)

    (
        country_table,
        industry_table,
        country_industry_table
    ) = build_distribution_tables(conn)

    (
        legal_table,
        sales_org_table
    ) = build_legal_sales_tables(conn)

    (
        customer_group_table,
        customer_nature_table,
        customer_hierarchy_table
    ) = build_customer_tables(conn)

    (
        sap_city_legal_table,
        sap_country_city_table,
        sap_sales_org_table
    ) = build_sap_tables(conn)

    (
        luci_country_gers_table,
        luci_city_legal_table,
        luci_country_city_table
    ) = build_luci_tables(conn)

    groups = [

        (
            "DATA QUALITY & PRIMARY KEY ANALYSIS",
            [
                ("DUPLICATE ANALYSIS", duplicate_table),
                ("PRIMARY KEY QUALITY", pk_table)
            ]
        ),

        (
            "LEGAL STATUS & SALES ORGANIZATION ANALYSIS",
            [
                ("LEGAL STATUS DISTRIBUTION", legal_table),
                ("SALES ORGANIZATION DISTRIBUTION", sales_org_table)
            ]
        ),

        (
            "GEOGRAPHIC & INDUSTRY DISTRIBUTION ANALYSIS",
            [
                ("COUNTRY VS INDUSTRY", country_industry_table),
                ("COUNTRY DISTRIBUTION", country_table),
                ("INDUSTRY DISTRIBUTION", industry_table)
            ]
        ),

        (
            "CUSTOMER SEGMENTATION & HIERARCHY ANALYSIS",
            [
                ("CUSTOMER HIERARCHY ANALYSIS", customer_hierarchy_table),
                ("CUSTOMER GROUP DISTRIBUTION", customer_group_table),
                ("CUSTOMER NATURE DISTRIBUTION", customer_nature_table)
            ]
        ),

        (
            "SAP MASTER DATA ANALYSIS",
            [
                ("SAP CITY VS LEGAL NAME", sap_city_legal_table),
                ("SAP COUNTRY VS CITY", sap_country_city_table),
                ("SAP SALES ORGANIZATION", sap_sales_org_table)
            ]
        ),

        (
            "LUCI MASTER DATA ANALYSIS",
            [
                ("LUCI COUNTRY VS GERS", luci_country_gers_table),
                ("LUCI CITY VS LEGAL NAME", luci_city_legal_table),
                ("LUCI COUNTRY VS CITY", luci_country_city_table)
            ]
        )

    ]

    row = 5

    for heading, tables in groups:

        row = write_group(
            row,
            heading,
            tables
        )

    for col in range(1, ws.max_column + 1):

        letter = get_column_letter(col)

        max_len = 0

        for r in range(1, ws.max_row + 1):

            value = ws.cell(r, col).value

            if value:

                max_len = max(
                    max_len,
                    len(str(value))
                )

        ws.column_dimensions[letter].width = min(
            max(max_len + 2, 12),
            28
        )

    wb.save(file_path)


business_understanding_framework(cursor, conn)
print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

Execution Time: 0.33 seconds


# Step - 6 : About Project Sheet

## 6.1 Build and Save Project Overview

In [106]:
start_time = time.perf_counter()
if "Business_Framework" in wb.sheetnames:
    ws_bf = wb["Business_Framework"]
    ws_bf.auto_filter.ref = None

overview_name = "Project_Overview"
if overview_name in wb.sheetnames:
    del wb[overview_name]

ws = wb.create_sheet(title=overview_name, index=0)
ws.sheet_view.showGridLines = False

header_fill = PatternFill(fill_type="solid", start_color="1F3A5F", end_color="1F3A5F")
section_fill = PatternFill(fill_type="solid", start_color="EAF2FB", end_color="EAF2FB")
white_fill = PatternFill(fill_type="solid", start_color="FFFFFF", end_color="FFFFFF")

header_font = Font(name="Calibri", size=18, bold=True, color="FFFFFF")
section_font = Font(name="Calibri", size=12, bold=True, color="1A1A1A")
body_font = Font(name="Calibri", size=11, color="1A1A1A")

center = Alignment(horizontal="center", vertical="center", wrap_text=True)
left = Alignment(horizontal="left", vertical="top", wrap_text=True)

ws.merge_cells("A1:H2")
ws["A1"] = "PHARMA BUSINESS UNDERSTANDING FRAMEWORK - MVP / POC"
ws["A1"].fill = header_fill
ws["A1"].font = header_font
ws["A1"].alignment = center

content = [
    (
        "Project Objective",
        "Construct a Business Understanding Framework for pharma master data that combines multiple sources, clarifies relationships between columns, and provides a business-readable summary for analysis and decisions.",
    ),
    (
        "Problem We Are Solving",
        "Pharma data is spread across systems and often contains missing values, duplicate records, invalid formats, and mismatched mappings between systems (for example SAP and LUCI). This creates inconsistent reporting, weak trust in analytics, and slower business decisions.",
    ),
    (
        "How This Project Solves The Problem",
        "The project standardizes and evaluates data through a structured framework: it profiles columns, validates keys and formats, checks cross-system consistency, and organizes outputs into business-aligned sections. This creates one clear view of data quality and business relationships so issues are visible and actionable.",
    ),
    (
        "Relationship And Column Analysis",
        "The framework examines how identifiers, legal entities, sales organizations, geography fields, and hierarchy attributes are connected. It summarizes what each column represents, how tables link through key business fields, and where relationships break due to quality issues.",
    ),
    (
        "Tools Used",
        "Python, Pandas, SQLite, and OpenPyXL are used for data ingestion, profiling, SQL-style relationship checks, quality rule evaluation, and Excel reporting presentation.",
    ),
    (
        "How Automation Is Implemented",
        "A repeatable pipeline is used: source loading, column profiling, relationship checks, quality checks, business-table assembly, and workbook publishing. This supports consistent reruns with updated source data while preserving the same analysis structure.",
    ),
    (
        "Output Sheets",
        "1) Data Quality Checks: KPI view and quality summaries across completeness, uniqueness, validity, and consistency.\n2) Business_Framework: grouped relationship analysis across Foundation, Legal & Sales, Distribution, Customer, SAP, and LUCI.",
    ),
]

row = 4

for title, text in content:

    # =====================================================
    # LEFT SIDE - QUESTION
    # =====================================================

    ws.merge_cells(
        start_row=row,
        start_column=1,
        end_row=row + 2,
        end_column=3
    )

    c_title = ws.cell(row=row, column=1)

    c_title.value = title

    c_title.fill = section_fill

    c_title.font = section_font

    c_title.alignment = left

    # =====================================================
    # RIGHT SIDE - ANSWER
    # =====================================================

    ws.merge_cells(
        start_row=row,
        start_column=4,
        end_row=row + 2,
        end_column=8
    )

    c_body = ws.cell(row=row, column=4)

    c_body.value = text

    c_body.fill = white_fill

    c_body.font = body_font

    c_body.alignment = left

    row += 4

for col in "ABCDEFGH":
    ws.column_dimensions[col].width = 24

ws.row_dimensions[1].height = 30
ws.row_dimensions[2].height = 30
for r in range(4, row + 1):
    ws.row_dimensions[r].height = 24

wb.save(file_path)

print("Project overview output file:", os.path.abspath(file_path))
print(f"Execution Time: {time.perf_counter() - start_time:.2f} seconds")

Project overview output file: c:\Users\U1109200\Downloads\INTERN_TASK_REPO\FrameWork\output_files\QualityCheck_results.xlsx
Execution Time: 0.07 seconds
